# W01-0b · Pandas basics *(before class, ~25 min)*

**pandas** is the Python library for tables. In class, W01-A loads 30,000 reviews into a pandas table and runs your
regexes over a whole column at once, and W01-B scores your patterns on a table of 300 support messages. This notebook
covers the pandas you need for that, one idea at a time: run each demo, predict the output before you look, then do the
short ✍️ exercise underneath. Replace `...` with your code; each exercise checks itself.

We practise on a made-up table of eight reviews, small enough to check every answer by eye. The last section loads the real data.

> **Already use pandas?** Skim the cheat sheet at the end and do the five exercises. If they pass, you're ready.

In [ ]:
import pandas as pd   # pandas, under its usual short name pd

# A DataFrame built from a dict: each key is a column name, each list holds that column's values, one per row
reviews = pd.DataFrame({
    "product": ["fridge", "fridge", "kettle", "kettle", "kettle", "dryer", "dryer", "fridge"],
    "rating":  [5, 1, 4, 2, 5, 1, 3, 4],
    "text": [
        "Great fridge, very quiet. Paid $899 and worth it.",
        "Stopped cooling after 3 months.<br />Service never came.",
        "Boils fast. The lid is a bit stiff.",
        "Started leaking after 2 weeks. Returned it.",
        "Perfect kettle for $25!",
        "Belt snapped after 6 months.<br />Very noisy before that.",
        "Does the job. Loud when spinning.",
        "Ice maker works well. Leak-free so far.",
    ],
})
reviews   # the last line of a cell is displayed: Jupyter shows a DataFrame as a table

## 1 · DataFrames, Series and the index

A **DataFrame** is a table. Each column is a **Series**: one column of values plus the row labels.
The row labels are the **index**, the bold numbers on the left. Here they are 0 to 7, but they are labels, not
positions: they stay attached to their rows when you filter or sort (section 4).

| | |
|---|---|
| `df.shape` | (rows, columns) |
| `len(df)` | number of rows |
| `df.columns` | the column names |
| `df.dtypes` | the type of each column: `int64` numbers, `bool` True/False, text shows as `object` (pandas 2) or `str` (pandas 3) |
| `df.head(n)` · `df.tail(n)` | the first · last `n` rows (default 5) |
| `df.sample(n, random_state=1)` | `n` random rows; the same `random_state` gives the same rows every time |

In [ ]:
print(reviews.shape)          # (8, 3): 8 rows, 3 columns
print(len(reviews))           # 8
print(list(reviews.columns))  # the column names; list() prints them as a plain Python list
print(reviews.dtypes)         # int64 for rating; the text columns show as object (pandas 2) or str (pandas 3)
reviews.head(3)               # the first 3 rows

## 2 · Selecting columns

- `df["col"]` gives **one column, as a Series**.
- `df[["col1", "col2"]]` (a *list* of names, hence the double brackets) gives **a smaller DataFrame**.
- `df.col` also works for simple names, but `df["col"]` always works, so this course uses it.

Assigning to a column name that doesn't exist yet creates a new column.

In [ ]:
print(reviews["rating"])                # a Series: row labels on the left, then the values, then its name and dtype
reviews["pct"] = reviews["rating"] * 20   # arithmetic on a column works on every row at once: 4 stars -> 80
reviews[["product", "rating", "pct"]]     # a DataFrame with just these three columns

Operations on a whole column at once, with no `for` loop, are called **vectorised**. They're shorter and much
faster than looping over rows, and nearly everything in this notebook works that way.

In [ ]:
reviews = reviews.drop(columns="pct")       # drop returns a new DataFrame without the column; we keep that one
list(reviews.columns)                       # back to the original three columns

## 3 · Filtering rows with True/False masks

Comparing a column with a value gives a Series of True/False, one per row, called a **mask**.
Putting a mask inside `[ ]` keeps the rows where it is True.

| | |
|---|---|
| `==` `!=` `<` `<=` `>` `>=` | compare every value in a column |
| `m1 & m2` · `m1 \| m2` · `~m` | and · or · not. **Put each comparison in parentheses**: `(a > 1) & (b < 3)` |
| `col.isin([...])` | is the value one of these? |
| `mask.sum()` · `mask.mean()` | how many are True · what **share** are True (True counts as 1, False as 0) |

Python's own `and`, `or` and `not` don't work on masks. Use `&`, `|` and `~`.

In [ ]:
low = reviews["rating"] <= 2                        # compare every rating with 2: one True/False per review
print(low)                                          # the mask itself
print(low.sum(), "low ratings, a share of", low.mean())   # sum counts the Trues; mean is the share of Trues

reviews[low]                                        # only the rows where the mask is True

In [ ]:
# Combine masks: fridges with a low rating. Without the parentheses, & would run before the comparisons.
reviews[(reviews["product"] == "fridge") & (reviews["rating"] <= 2)]

In [ ]:
reviews[~reviews["product"].isin(["fridge", "dryer"])]   # ~ flips the mask: everything except fridges and dryers

### Rows *and* columns at once: `.loc`

`df.loc[rows, columns]` picks rows and columns in one step. The rows can be a mask, one label or a list of labels,
and `:` means all. `df.at[label, "col"]` gets a single value.

In [ ]:
# .loc[rows, columns] picks rows AND columns in one step:
#   rows:    reviews["rating"] == 5, a mask, so only the rows where it is True (a label or a list of labels also works)
#   columns: a list of column names; write : instead to keep every column
print(reviews.at[2, "text"])                               # .at[label, column]: exactly one cell, row label 2, column "text"
reviews.loc[reviews["rating"] == 5, ["product", "text"]]   # the 5-star rows, only the product and text columns

### ✍️ Exercise 1: the unhappy customers

Select the `product` and `text` of every review rated **2 or lower**, using `.loc`.

In [ ]:
# Hint: reviews.loc[<a mask for the rows>, <a list of column names>]
unhappy = ...   # ✍️
print(unhappy)
# ✅ check: the row labels tell us which reviews you kept, the column names which columns
assert list(unhappy.index) == [1, 3, 5], "not quite: check which rows you kept"
assert list(unhappy.columns) == ["product", "text"], "right rows; now keep only the product and text columns"
print("✅")

## 4 · Labels vs positions: `.loc` and `.iloc`

A filtered table keeps the **original row labels**. That's useful, because you can always look a row up in the full
table again, but it means "row 0" is ambiguous:

- `.loc[...]` uses **labels** (the index).
- `.iloc[...]` uses **positions**, 0 to `len(df) - 1`, like a Python list.

In [ ]:
five_star = reviews[reviews["rating"] == 5]   # a filtered table: only rows 0 and 4 survive
print(list(five_star.index))            # [0, 4]: the labels came along

print(five_star.iloc[1]["text"])        # position 1: the second row, whatever its label
print(five_star.loc[4, "text"])         # label 4: the same row
# five_star.loc[1, "text"] would fail with a KeyError: no row in five_star has the label 1

When pandas combines two Series, it **matches them up by label**, not by position. So a column computed from a
filtered table can be put back next to the full table, and each value lands on the row it came from.
W01-A relies on this in section 4.

In [ ]:
lengths = five_star["text"].str.len()   # characters per review (section 6), only for the 5-star rows
reviews["length"] = lengths             # lines up by label; the other rows get NaN, "no value"
reviews

In [ ]:
reviews = reviews.drop(columns="length")   # remove the demo column again

**Change a table with `.loc`, not with two sets of brackets.** `df.loc[mask, "col"] = value` changes the rows you
selected. `df[mask]["col"] = value` looks similar, but it changes a temporary copy and leaves `df` as it was, with
at most a warning.

## 5 · Counting and summarising

| | |
|---|---|
| `s.value_counts()` | how many times each value appears, most common first |
| `s.value_counts(normalize=True)` | the same as shares that add up to 1 |
| `s.sort_index()` · `s.sort_values()` | sort by label · by value (`ascending=False` for largest first) |
| `s.mean()` `s.sum()` `s.min()` `s.max()` `s.nunique()` | one number for the whole column |
| `df.describe()` | count, mean, spread and quartiles of every number column |
| `s.round(3)` | round to 3 decimals |

Each method returns a new object, so you can **chain** them one after another. Wrap a long chain in parentheses
and put one step per line, with a comment on each.

In [ ]:
print(reviews["product"].value_counts())   # how many reviews per product, most common first
print(reviews["product"].nunique(), "different products; average rating", reviews["rating"].mean())

(reviews["rating"]
    .value_counts(normalize=True)   # share of reviews at each rating
    .sort_index()                   # in rating order 1..5, not most-common-first
    .round(3))                      # 3 decimals

### ✍️ Exercise 2: share of 5-star reviews

What share of the reviews are rated exactly 5? Use a mask and `.mean()`.

In [ ]:
# Hint: build a mask like `low` in section 3, then take its .mean() (True counts as 1, False as 0)
share5 = ...   # ✍️
print(share5)
assert share5 == 0.25, "not quite: 2 of the 8 reviews are 5-star"
print("✅")

## 6 · Text columns: the `.str` methods

A text column has a `.str` accessor that runs a string method **on every row**. Most of them accept a regex, which
is where W01-0 and this notebook meet.

| | returns, per row |
|---|---|
| `.str.lower()` · `.str.len()` | lower-cased text · number of characters |
| `.str.contains(p, case=False)` | True/False: does the text match `p` anywhere? (a mask) |
| `.str.replace(p, new, regex=True)` | the text with every match replaced. Without `regex=True`, `p` is plain text |
| `.str.findall(p)` | a **list** of every match |
| `.str.extract(p)` | one column per capturing group: the first match's groups, or NaN |
| `.str.split()` | a list of the words (split at whitespace) |

When a row's value is a list, `.str.len()` gives the length of the list and `.explode()` turns each list item into a row of its own.

In [ ]:
reviews["clean"] = reviews["text"].str.replace(r"<br\s*/?>", " ", regex=True)   # <br /> tags become spaces
leaks = reviews["clean"].str.contains(r"leak", case=False)                       # True/False per review
reviews.loc[leaks, ["clean"]]   # .loc again: the rows that mention "leak", only the clean column. Note "Leak-free": a false positive

In [ ]:
prices = reviews["clean"].str.findall(r"\$\d+")   # a list per review; [] where there's no price
print(prices)
print(prices.explode().dropna())                   # one row per price; the [] rows became NaN, so drop them

In [ ]:
# Two capturing groups -> two columns, named 0 and 1; NaN where the review doesn't match
after = reviews["clean"].str.extract(r"after (\d+) (day|week|month)s?")
after.columns = ["n", "unit"]   # give the columns names
after.dropna()                  # only the reviews that matched, with their original labels

**NaN** ("not a number") is how pandas marks a missing value. `.dropna()` drops rows with one, `.isna()` is a
mask of where they are, and `.fillna(x)` fills them in. Most calculations, such as `.mean()`, skip them.

### ✍️ Exercise 3: words per review

Add a column `n_words` with the number of words in each review's `clean` text. Chain two `.str` methods.

In [ ]:
# Hint: first split each text into a list of words, then take the length of each list
reviews["n_words"] = ...   # ✍️
print(reviews[["clean", "n_words"]])
assert list(reviews["n_words"]) == [9, 8, 8, 7, 4, 9, 6, 7], "not quite: split into words first, then count them"
print("✅")

## 7 · Your own function on every row: `.map`, `.astype` and `.apply`

- `s.map(d)` replaces each value by looking it up in the dict `d` (values not in `d` become NaN).
- `s.astype(float)` converts the type, for example the text `"3"` that `.str.extract` returns into the number `3.0`.
- `s.apply(f)` calls the function `f` on each value and collects the results.

For a function you only need once, write a **lambda**: `lambda x: x * 2` is a small unnamed function that takes `x` and returns `x * 2`.

In [ ]:
size = {"fridge": "large", "dryer": "large", "kettle": "small"}   # product -> size
print(reviews["product"].map(size).tolist())                       # look each product up in the dict; tolist() prints a plain list

print(after["n"].astype(float).tolist())                          # text -> numbers (NaN stays NaN)

print(reviews["clean"].apply(lambda t: t.split(".")[0]).tolist())   # the first sentence of each review

### ✍️ Exercise 4: star ratings as sentiment labels

Add a column `sentiment`: `"positive"` for 4 or 5 stars, `"negative"` for 1 or 2, `"neutral"` for 3.
Use `.apply` with a lambda (a one-line `if` looks like `a if condition else b`, and you can chain two of them).

In [ ]:
# Hint: lambda r: "positive" if ... else "negative" if ... else "neutral"
reviews["sentiment"] = reviews["rating"].apply(lambda r: ...)   # ✍️
print(reviews[["rating", "sentiment"]])
assert list(reviews["sentiment"]) == ["positive", "negative", "positive", "negative",
                                      "positive", "negative", "neutral", "positive"], "not quite"
print("✅")

## 8 · Groups: `groupby` and `crosstab`

`df.groupby("col")` splits the rows into one group per value of `col`. Pick a column and say what to compute per
group: `.mean()`, `.size()`, or several at once with `.agg(["count", "mean"])`. You can also group by a mask, which
gives a True group and a False group. W01-A uses this to ask *do reviews that mention X rate differently?*

`pd.crosstab(a, b)` counts how often each pair of values occurs: a table with the values of `a` as rows and `b` as columns.

In [ ]:
print(reviews.groupby("product")["rating"].mean())   # one group per product, then the mean rating of each

mentions_after = reviews["clean"].str.contains(r"\bafter \d+")   # True/False: does the review say "after <number>"?
# The groups are False (no "after N") and True; the index is named "clean", after the column the mask came from
reviews.groupby(mentions_after)["rating"].agg(["count", "mean"])   # per group: how many reviews and their average rating

In [ ]:
pd.crosstab(reviews["product"], reviews["sentiment"])   # rows: product · columns: sentiment · cells: how many reviews

### ✍️ Exercise 5: best product

Build a table with one row per product and two columns, the `count` and `mean` of its ratings, sorted so the
**highest mean comes first**. (`sort_values("mean", ascending=False)` sorts a DataFrame by one of its columns.)

In [ ]:
# Hint: chain three steps: groupby the product, .agg the count and mean of the ratings, sort by the mean
by_product = ...   # ✍️
print(by_product)
assert list(by_product.index) == ["kettle", "fridge", "dryer"], "not quite: check the sort order"
assert list(by_product.columns) == ["count", "mean"], "right order; now show both count and mean"
print("✅")

## 9 · The real data

`pd.read_csv` reads a CSV file into a DataFrame. It decompresses `.gz` files itself, based on the extension.
The data folder sits next to `w01/`, so from this notebook its path is `../data/`.

Two options W01-B uses: `dtype=str` keeps every column as text, exactly as written (so a PIN code like `011017`
keeps its leading 0), and `keep_default_na=False` reads an empty cell as `""` instead of NaN.

In [ ]:
pd.set_option("display.max_colwidth", 160)   # show 160 characters per cell instead of 50, so text isn't cut off

real = pd.read_csv("../data/reviews.csv.gz")   # the data folder is next to w01/; pandas unzips the .gz itself
print(real.shape)                              # (30000, 9)
print(list(real.columns))
real[["rating", "title", "text"]].sample(3, random_state=1)   # 3 random reviews, 3 columns

In [ ]:
print(real["parent_asin"].nunique(), "products")   # parent_asin is the product ID
real["text"].str.len().describe().round(0)   # how long are the reviews, in characters?

Try a few things on `real` before class: which ratings are most common? How many reviews contain `<br`?
What's the average rating of reviews that mention `"return"`? Everything above works unchanged on 30,000 rows.

In [ ]:
# your experiments here

## Cheat sheet

| | |
|---|---|
| **Look** | `df.shape` · `len(df)` · `df.columns` · `df.dtypes` · `df.head()` · `df.sample(n, random_state=1)` |
| **Columns** | `df["col"]` a Series · `df[["a", "b"]]` a DataFrame · `df["new"] = …` add · `df.drop(columns="col")` |
| **Rows** | `df[mask]` · `(m1) & (m2)` · `\|` · `~m` · `.isin([...])` · `mask.sum()` count · `mask.mean()` share |
| **Pick** | `df.loc[rows, cols]` by label · `df.iloc[i]` by position · `df.at[label, "col"]` one value |
| **Summarise** | `value_counts(normalize=True)` · `sort_index()` · `sort_values(ascending=False)` · `mean()` · `nunique()` · `describe()` · `round(n)` |
| **Text** | `.str.lower()` · `.str.len()` · `.str.contains(p, case=False)` · `.str.replace(p, new, regex=True)` · `.str.findall(p)` · `.str.extract(p)` · `.str.split()` · `.explode()` |
| **Transform** | `.map(dict)` · `.astype(float)` · `.apply(lambda x: …)` · `.dropna()` · `.isna()` · `.fillna(x)` |
| **Groups** | `df.groupby("col")["x"].agg(["count", "mean"])` · `pd.crosstab(a, b)` |
| **Load** | `pd.read_csv(path, dtype=str, keep_default_na=False)` · `pd.set_option("display.max_colwidth", 160)` |

➡️ In class, **W01-A** combines this with your regexes on all 30,000 reviews.